In [ ]:
import argparse
import pathlib
import random
from dataclasses import dataclass
from typing import Any, Callable

import joblib
import mlflow
import numpy as np
import optuna
import torch
from models.UNet import UNet

from callbacks.CallbackPipeline import CallbackPipeline
from callbacks.utils.SampleImages import SampleImages
from callbacks.utils.SaveEpochCrops import SaveEpochCrops
from datasets.dataset_00.CellCropToCropDataset import CellCropToCropDataset
from datasets.dataset_00.utils.CropCacheBuilder import (
    ensure_dapi_to_gold_cache, load_cache_manifest)
from datasets.dataset_00.utils.ImagePostProcessor import ImagePostProcessor
from datasets.dataset_00.utils.ImagePreProcessor import ImagePreProcessor
from losses.L1Loss import L1Loss
from metrics.L1 import L1
from metrics.L2 import L2
from metrics.PSNR import PSNR
from metrics.SSIM import SSIM
from splitters.HashSplitter import HashSplitter
from trainers.UNetTrainer import UNetTrainer

In [ ]:
@dataclass(frozen=True)
class DatasetConfig:
    image_dir: pathlib.Path
    parquet_path: pathlib.Path
    cache_root: pathlib.Path
    input_channel: str
    target_channel: str
    metadata_column_map: dict[str, str] | None = None


DATASET_CONFIGS = {
    "u2os": DatasetConfig(
        image_dir=pathlib.Path(
            "/mnt/big_drive/nuclear_speckle_data/u20s_dataset_jan_15_2026/u20s_images/tiffs"
        ),
        parquet_path=pathlib.Path(
            "/mnt/big_drive/nuclear_speckle_data/u20s_dataset_jan_15_2026/u20s_profiles/single_cell_profiles/u2os_per_nuclei_sc_feature_selected.parquet"
        ),
        cache_root=pathlib.Path(
            "/mnt/big_drive/nuclear_speckle_data/u20s_dataset_jan_15_2026/model_cache"
        ),
        input_channel="CH01",
        target_channel="CH03",
        metadata_column_map={
            "Metadata_Position": "Metadata_Site",
        },
    ),
    "initial": DatasetConfig(
        image_dir=pathlib.Path(
            "/mnt/big_drive/nuclear_speckle_data/initial_dataset/IC_corrected_images"
        ),
        parquet_path=pathlib.Path(
            "/mnt/big_drive/nuclear_speckle_data/initial_dataset/Preprocessed_data/cleaned_sc_profiles"
        ),
        cache_root=pathlib.Path(
            "/mnt/big_drive/nuclear_speckle_data/initial_dataset/model_cache"
        ),
        input_channel="CH0",
        target_channel="CH2",
        metadata_column_map={
            "Image_Metadata_Plate": "Metadata_Plate",
            "Image_Metadata_Well": "Metadata_Well",
            "Image_Metadata_Site": "Metadata_Site",
            "Nuclei_AreaShape_BoundingBoxMinimum_X": "Metadata_Nuclei_AreaShape_BoundingBoxMinimum_X",
            "Nuclei_AreaShape_BoundingBoxMaximum_X": "Metadata_Nuclei_AreaShape_BoundingBoxMaximum_X",
            "Nuclei_AreaShape_BoundingBoxMinimum_Y": "Metadata_Nuclei_AreaShape_BoundingBoxMinimum_Y",
            "Nuclei_AreaShape_BoundingBoxMaximum_Y": "Metadata_Nuclei_AreaShape_BoundingBoxMaximum_Y",
        },
    ),
}


parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=20)
parser.add_argument("--n-trials", type=int, default=4)
parser.add_argument("--max-train-batches", type=int, default=-1)
parser.add_argument("--max-eval-batches", type=int, default=-1)
parser.add_argument("--enable-image-savers", type=int, choices=[0, 1], default=1)
parser.add_argument("--dataset", choices=sorted(DATASET_CONFIGS.keys()), default="u2os")
args = parser.parse_args([])

max_train_batches = None if args.max_train_batches <= 0 else args.max_train_batches
max_eval_batches = None if args.max_eval_batches <= 0 else args.max_eval_batches

In [ ]:
class OptimizationManager:
    """Optuna objective function with MLflow logging."""

    def __init__(
        self,
        trainer: Any,
        hash_splitter: Any,
        dataset: Any,
        callbacks_args: dict[str, Any],
        model_factory: Callable[[], torch.nn.Module],
        **trainer_kwargs,
    ):
        self.trainer = trainer
        self.hash_splitter = hash_splitter
        self.dataset = dataset
        self.callbacks_args = callbacks_args
        self.model_factory = model_factory
        self.trainer_kwargs = trainer_kwargs

    def __call__(self, trial: optuna.trial.Trial):
        batch_size = trial.suggest_int("batch_size", 8, 32)
        lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)

        train_dataloader, val_dataloader, _ = self.hash_splitter(batch_size=batch_size)
        self.trainer_kwargs["train_dataloader"] = train_dataloader
        self.trainer_kwargs["val_dataloader"] = val_dataloader

        model = self.model_factory()
        self.trainer_kwargs["model"] = model

        optimizer_params = {
            "params": model.parameters(),
            "lr": lr,
            "betas": (0.5, 0.999),
        }

        loss_trainer = L1Loss()
        loss_callbacks = L1(device=device)
        metrics = [
            L2(device=device),
            PSNR(device=device, max_pixel_value=1.0),
            SSIM(device=device, max_pixel_value=1.0),
        ]

        with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
            optimizer = torch.optim.Adam(**optimizer_params)
            self.trainer_kwargs["model_optimizer"] = optimizer

            opt_params = optimizer.param_groups[0].copy()
            del opt_params["params"]
            mlflow.log_params({f"optimizer_{k}": v for k, v in opt_params.items()})
            mlflow.log_param("batch_size", batch_size)
            mlflow.set_tag("optimizer_class", optimizer.__class__.__name__.lower())

            self.trainer_kwargs["callbacks"] = CallbackPipeline(
                **self.callbacks_args | {"metrics": metrics, "loss": loss_callbacks}
            )

            trainer_obj = self.trainer(
                **self.trainer_kwargs | {"model_loss": loss_trainer}
            )
            trainer_obj.train()

            return trainer_obj.best_loss_value

In [ ]:
dataset_config = DATASET_CONFIGS[args.dataset]
image_dir = dataset_config.image_dir.resolve(strict=True)
parquet_path = dataset_config.parquet_path.resolve(strict=True)
cache_root = dataset_config.cache_root
crop_cache_path = cache_root / "dapi_to_gold_crop_cache"
tensor_cache_path = cache_root / "paired_tensor_cache"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
mlflow.log_param("random_seed", 0)
mlflow.log_param("dataset", args.dataset)
mlflow.log_param("input_channel", dataset_config.input_channel)
mlflow.log_param("target_channel", dataset_config.target_channel)

description = """
Optimization of a DAPI-to-Gold image-to-image translation model with:
- UNet Generator
- Single 2D crop input and single 2D crop target
- Cache-backed filtered nucleus crops generated from the configured data directory
- L1 optimization objective with L2, PSNR, and SSIM metric logging
"""
mlflow.set_tag("mlflow.note.content", description)

cache_result = ensure_dapi_to_gold_cache(
    image_dir=image_dir,
    parquet_path=parquet_path,
    cache_dir=crop_cache_path,
    input_channel=dataset_config.input_channel,
    target_channel=dataset_config.target_channel,
    metadata_column_map=dataset_config.metadata_column_map,
)
manifest_rows = load_cache_manifest(manifest_path=cache_result.manifest_path)
image_specs = cache_result.image_specs

mlflow.log_param("input_max_pixel_value", image_specs["input_max_pixel_value"])
mlflow.log_param("target_max_pixel_value", image_specs["target_max_pixel_value"])

image_preprocessor = ImagePreProcessor(image_specs=image_specs, device=device)
image_postprocessor = ImagePostProcessor()

crop_image_dataset = CellCropToCropDataset(
    manifest_rows=manifest_rows,
    image_specs=image_specs,
    image_preprocessor=image_preprocessor,
    image_cache_path=tensor_cache_path,
)

hash_splitter = HashSplitter(
    dataset=crop_image_dataset,
    train_frac=0.8,
    val_frac=0.1,
)

_, val_dataloader, _ = hash_splitter(batch_size=16)
crop_dataset_idxs = SampleImages(datastruct=val_dataloader, image_fraction=1 / 32)()

image_prediction_saver = SaveEpochCrops(
    image_dataset=val_dataloader.dataset.dataset,
    image_postprocessor=image_postprocessor,
    image_dataset_idxs=crop_dataset_idxs,
)

callbacks_args = {
    "early_stopping_counter_threshold": 5,
    "image_savers": [image_prediction_saver] if args.enable_image_savers == 1 else None,
    "image_postprocessor": image_postprocessor,
    "max_eval_batches": max_eval_batches,
}

In [ ]:
optimization_manager = OptimizationManager(
    trainer=UNetTrainer,
    hash_splitter=hash_splitter,
    dataset=crop_image_dataset,
    callbacks_args=callbacks_args,
    model_factory=lambda: UNet(in_channels=1, out_channels=1),
    epochs=args.epochs,
    max_train_batches=max_train_batches,
)

study = optuna.create_study(study_name="model_training", direction="minimize")
study.optimize(optimization_manager, n_trials=args.n_trials)

joblib.dump(study, "optuna_study.joblib")
mlflow.log_artifact("optuna_study.joblib")